
# Endogenous Reachability Collapse (ERC) Model

**Working title:** *Endogenous Reachability Collapse (ERC): A Minimal Dynamical Model for Loss of Recoverability Without Loss of State Space*

This notebook contains the cleaned, reproducible version of the ERC model developed in conversation. It is organized for **GitHub publication** and includes:

- model definition
- frozen-constraint geometry
- endogenous simulations
- representative trajectories
- a brief interpretation and limits section

## Core hypothesis

A system may preserve its underlying state space and still lose access to a base attractor because an **internal slow constraint** shrinks the recoverable region over time.

## Important scope note

This notebook supports a **reachability-collapse** result. It does **not** yet claim full **trajectory exclusion** in the stronger topological sense. In other words:

- **shown here:** endogenous loss of recoverability / reachability collapse
- **not yet shown here:** elimination of return trajectories as valid solutions of the full dynamics

That distinction should remain explicit in any public writeup.



## Model definition

We define a slow–fast system with two fast variables \(x(t), y(t)\) and one slow internal constraint \(c(t)\).

\[
r^2 = x^2 + y^2
\]

\[
\dot{x} = g(r,c)x - \omega y
\]
\[
\dot{y} = g(r,c)y + \omega x
\]

with

\[
g(r,c) = -\left(r^2 - a(c)^2\right)\left(r^2 - b^2\right)
\]

and a slow internal variable

\[
\dot{c} = \varepsilon \left(\frac{r^2}{1+r^2} - c\right)
\]

where the recoverability boundary is

\[
a(c) = a_0 - \alpha c
\]

with a small positive floor so that \(a(c)\) does not become negative.

### Interpretation

For fixed \(c\):

- \(r = 0\) is the **base attractor**
- \(r = a(c)\) is an **unstable boundary**
- \(r = b\) is an **outer attractor**

As \(c\) increases, \(a(c)\) shrinks, which contracts the recoverable basin of the base attractor.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ----------------------------
# Parameters
# ----------------------------
omega = 1.5     # angular speed in phase space
b = 1.20        # outer stable cycle radius
a0 = 0.75       # unstable boundary at c=0
alpha = 0.55    # how much constraint shrinks the recovery basin
eps = 0.03      # slow timescale for c
a_min = 0.05    # keep a(c) positive

# Plotting helper
theta = np.linspace(0, 2*np.pi, 400)


In [ ]:

def a_of_c(c):
    return max(a_min, a0 - alpha * c)

def rhs_full(t, z):
    x, y, c = z
    r2 = x*x + y*y
    a = a_of_c(c)
    g = -((r2 - a*a) * (r2 - b*b))
    dx = g*x - omega*y
    dy = g*y + omega*x
    dc = eps * (r2 / (1.0 + r2) - c)
    return [dx, dy, dc]

def rhs_frozen_c(c_fixed):
    def f(t, z):
        x, y = z
        r2 = x*x + y*y
        a = a_of_c(c_fixed)
        g = -((r2 - a*a) * (r2 - b*b))
        dx = g*x - omega*y
        dy = g*y + omega*x
        return [dx, dy]
    return f

def classify_return(r, r_return=0.12, tail_fraction=0.2):
    n = len(r)
    tail = r[int((1 - tail_fraction) * n):]
    return np.mean(tail) < r_return


## Frozen-constraint geometry

In [ ]:

def simulate_frozen_fast(z0_xy, c_fixed, Tfast=80, npts=600):
    t_eval_fast = np.linspace(0, Tfast, npts)
    sol = solve_ivp(
        rhs_frozen_c(c_fixed),
        [0, Tfast],
        z0_xy,
        t_eval=t_eval_fast,
        rtol=1e-5,
        atol=1e-7
    )
    x, y = sol.y
    r = np.sqrt(x*x + y*y)
    return sol.t, x, y, r

def basin_map_fast(c_fixed, grid_lim=1.6, N=20):
    xs = np.linspace(-grid_lim, grid_lim, N)
    ys = np.linspace(-grid_lim, grid_lim, N)
    basin = np.zeros((N, N))

    for i, x0 in enumerate(xs):
        for j, y0 in enumerate(ys):
            t, x, y, r = simulate_frozen_fast([x0, y0], c_fixed)
            basin[j, i] = 1 if classify_return(r) else 0

    return xs, ys, basin


In [ ]:

# Example phase-space trajectories at c=0
def simulate_full(z0, T=250, npts=4000):
    t_eval = np.linspace(0, T, npts)
    sol = solve_ivp(rhs_full, [0, T], z0, t_eval=t_eval, rtol=1e-8, atol=1e-10)
    x, y, c = sol.y
    r = np.sqrt(x*x + y*y)
    return sol.t, x, y, r, c

initial_conditions = [
    [0.20, 0.00, 0.00],
    [0.55, 0.00, 0.00],
    [0.90, 0.00, 0.00],
    [1.30, 0.00, 0.00],
]

plt.figure(figsize=(7, 7))
for z0 in initial_conditions:
    t, x, y, r, c = simulate_full(z0)
    plt.plot(x, y, lw=1.8, label=f"z0={tuple(np.round(z0,2))}")

a_ref = a_of_c(0.0)
plt.plot(a_ref*np.cos(theta), a_ref*np.sin(theta), "--", label=f"unstable boundary c=0: a={a_ref:.2f}")
plt.plot(b*np.cos(theta), b*np.sin(theta), ":", label=f"outer attractor: b={b:.2f}")
plt.scatter([0], [0], s=60, marker="x", label="base attractor")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Phase space trajectories (full system)")
plt.axis("equal")
plt.legend()
plt.show()


In [ ]:

# Basin maps for fixed c values
c_values = [0.0, 0.4, 0.8]

fig, axes = plt.subplots(1, len(c_values), figsize=(15, 5))

for ax, c_fixed in zip(axes, c_values):
    xs, ys, basin = basin_map_fast(c_fixed, N=20)

    ax.imshow(
        basin,
        origin="lower",
        extent=[xs.min(), xs.max(), ys.min(), ys.max()],
        aspect="equal"
    )

    a_now = a_of_c(c_fixed)
    ax.plot(a_now*np.cos(theta), a_now*np.sin(theta), "--", lw=1.5, label=f"a(c)={a_now:.2f}")
    ax.plot(b*np.cos(theta), b*np.sin(theta), ":", lw=1.5, label=f"b={b:.2f}")

    ax.set_title(f"Fast basin map (c={c_fixed:.1f})")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:

# Geometric approximation of recoverable fraction vs constraint
c_scan = np.linspace(0, 1.0, 101)
a_scan = np.array([a_of_c(c) for c in c_scan])

grid_lim = 1.6
fractions_geom = np.pi * a_scan**2 / (2 * grid_lim)**2

plt.figure(figsize=(6.5, 4.5))
plt.plot(c_scan, fractions_geom, "o-", markersize=3)
plt.xlabel("Constraint level c")
plt.ylabel("Approx. recoverable fraction")
plt.title("Geometric reachable fraction vs constraint")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

print("Example values:")
print("c_scan[:6] =", c_scan[:6])
print("fractions_geom[:6] =", fractions_geom[:6])



## Endogenous simulations

We now allow the internal constraint \(c(t)\) to evolve with the dynamics and test whether return to the base attractor is lost from some initial radii, even though the attractor itself remains present.


In [ ]:

def simulate_full_fast(z0, Tfast=120, npts=1200):
    t_eval_fast = np.linspace(0, Tfast, npts)
    sol = solve_ivp(
        rhs_full,
        [0, Tfast],
        z0,
        t_eval=t_eval_fast,
        rtol=1e-6,
        atol=1e-8
    )
    x, y, c = sol.y
    r = np.sqrt(x*x + y*y)
    return sol.t, x, y, r, c


In [ ]:

r0_values = np.linspace(0.05, 1.10, 16)

returned = []
final_c = []
min_margin = []
return_time = []

for r0 in r0_values:
    z0 = [r0, 0.0, 0.0]
    t, x, y, r, c = simulate_full_fast(z0)

    a_t = np.array([a_of_c(ci) for ci in c])
    margin = a_t - r

    tail = r[int(0.8 * len(r)):]
    is_return = np.mean(tail) < 0.12

    returned.append(is_return)
    final_c.append(c[-1])
    min_margin.append(np.min(margin))

    idx = np.where(r < 0.12)[0]
    if len(idx) > 0:
        return_time.append(t[idx[0]])
    else:
        return_time.append(np.nan)

print("r0_values =", np.round(r0_values, 3))
print("returned =", returned)
print("final_c =", np.round(final_c, 4))
print("min_margin =", np.round(min_margin, 4))
print("return_time =", np.round(return_time, 4))


In [ ]:

plt.figure(figsize=(6.5, 4.5))
plt.plot(r0_values, np.array(returned).astype(float), "o-")
plt.xlabel("Initial radius r0")
plt.ylabel("Returned to base attractor? (1=yes, 0=no)")
plt.title("Endogenous reachability from c(0)=0")
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(6.5, 4.5))
plt.plot(r0_values, final_c, "o-")
plt.xlabel("Initial radius r0")
plt.ylabel("Final constraint c(T)")
plt.title("Constraint accumulation vs initial radius")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(6.5, 4.5))
plt.plot(r0_values, min_margin, "o-")
plt.axhline(0, linestyle="--")
plt.xlabel("Initial radius r0")
plt.ylabel("min[a(c(t)) - r(t)]")
plt.title("Dynamic safety margin")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(6.5, 4.5))
plt.plot(r0_values, return_time, "o-")
plt.xlabel("Initial radius r0")
plt.ylabel("Return time")
plt.title("Return time vs initial radius")
plt.grid(True, alpha=0.3)
plt.show()



## Representative trajectories

These three trajectories provide the narrative core of the ERC model:

- one that returns clearly
- one near the regime boundary
- one that fails to return

For each, we plot:
- instantaneous radius \(r(t)\)
- moving recoverability boundary \(a(c(t))\)
- internal constraint \(c(t)\)
- dynamic safety margin \(a(c(t)) - r(t)\)


In [ ]:

selected_r0 = [0.40, 0.68, 0.82]
results = {}

for r0 in selected_r0:
    z0 = [r0, 0.0, 0.0]
    t, x, y, r, c = simulate_full_fast(z0)

    a_t = np.array([a_of_c(ci) for ci in c])
    margin = a_t - r

    results[r0] = {
        "t": t,
        "x": x,
        "y": y,
        "r": r,
        "c": c,
        "a_t": a_t,
        "margin": margin,
    }

print("Simulations ready.")


In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex='col')

for col, r0 in enumerate(selected_r0):
    t = results[r0]["t"]
    r = results[r0]["r"]
    c = results[r0]["c"]
    a_t = results[r0]["a_t"]
    margin = results[r0]["margin"]

    # Row 1
    axes[0, col].plot(t, r, label="r(t)")
    axes[0, col].plot(t, a_t, "--", label="a(c(t))")
    axes[0, col].set_title(f"r0 = {r0:.2f}")
    axes[0, col].set_ylabel("Radius / boundary")
    axes[0, col].grid(True, alpha=0.3)
    axes[0, col].legend()

    # Row 2
    axes[1, col].plot(t, c)
    axes[1, col].set_ylabel("c(t)")
    axes[1, col].grid(True, alpha=0.3)

    # Row 3
    axes[2, col].plot(t, margin)
    axes[2, col].axhline(0, linestyle="--")
    axes[2, col].set_ylabel("a(c)-r")
    axes[2, col].set_xlabel("time")
    axes[2, col].grid(True, alpha=0.3)

plt.suptitle("Endogenous Reachability Collapse (ERC)", fontsize=16)
plt.tight_layout()
plt.show()



## Interpretation

What this notebook supports:

1. The **base attractor remains present**.
2. The **recoverable basin contracts** as the internal constraint grows.
3. Return can fail because the system loses **reachability**, not because the state space vanishes.

A useful summary sentence is:

> The system does not lose the attractor. It loses access to it.

## Limit / next step

An important distinction remains between:

- **dynamic inaccessibility / reachability collapse**
- **structural impossibility / trajectory exclusion**

The current ERC notebook supports the first. A stronger next model would need to show that return trajectories are no longer valid solutions of the full dynamics, for example via a topological change in flow, bifurcation, or loss of dynamical connectivity.
